# First Program

## Setting up DSPy

### Load environment variables

In [1]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [2]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [3]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## Signatures

DSPy **Signature** describes the inputs a function accepts and the outputs it returns. It’s how we define our task. Similar to function signatures in programming.

In [8]:
haiku_signature = "subject -> haiku"

print("Type:", type(haiku_signature))

Type: <class 'str'>


To turn our Signature into a callable function, we use `dspy.Predict`. Predict is a kind of [DSPy `Module`](https://dspy.ai/diving-deeper/modules/).

- If Signatures specify _what_ we want
- **Modules** define _how_ we aim to achieve it. They implement a call-time strategy, manage the control flow, tools, and more.

![](../assets/signature_module.png){height=196}

In [10]:
haiku_generator = dspy.Predict(haiku_signature)

print("Type:   ", type(haiku_generator))
print("Module: ", isinstance(haiku_generator, dspy.Module))

Type:    <class 'dspy.predict.predict.Predict'>
Module:  True


Run the predictor module, passing inputs as per the signature:

In [11]:
result = haiku_generator(subject="computer science")
print(result.haiku)

Codes weave through the night,  
Logic dances in circuits,  
Logic sparks the dawn.


In just four lines we built an AI-powered program that reads like software and acts like a function.

## Where is the prompt?

In DSPy, to produce a prompt you compose a signature and the messages are written for you.

- DSPy manages all the prompting and templating.
- Call `dspy.inspect_history(n=1)` to take a look at the formatted prompt our program produced and the string the `lm` returned.

In [12]:
dspy.inspect_history(n=1)





[2026-06-16T07:14:35.469326]

System message:

Your input fields are:
1. `subject` (str):
Your output fields are:
1. `haiku` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## subject ## ]]
{subject}

[[ ## haiku ## ]]
{haiku}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `subject`, produce the fields `haiku`.


User message:

[[ ## subject ## ]]
computer science

Respond with the corresponding output fields, starting with the field `[[ ## haiku ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## haiku ## ]]
Codes weave through the night,  
Logic dances in circuits,  
Logic sparks the dawn.  
[[ ## completed ## ]]







## Expanding your signature: more inputs and adding types

### Adding additional inputs and outputs

Adding more fields to signature strings is as easy as separating field names with commas. For example, let’s update our program to accept two inputs, a `location` and `mood`:

In [13]:
haiku_bot = dspy.Predict("location, mood -> haiku")
result = haiku_bot(location="a quiet library", mood="mysterious")
print(result.haiku)

Shadows whisper now,
Pages hide secrets untold,
Mysteries unfold.


Defining additional outputs works identically:


In [14]:
haiku_bot = dspy.Predict("location, mood -> haiku, haiku_title")
result = haiku_bot(location="a quiet library", mood="mysterious")
print(result.haiku_title)
print("- - -")
print(result.haiku)

Whispers in the Margin
- - -
Silent pages whisper,
Shadows dance on quiet walls—
Secrets hide in ink.


## Hone your signature by mindfully naming your fields

The field names we choose aren’t just for our own readability. Unlike traditional programming, where variable names are purely identifiers, the LM reads them too, and uses them to infer what each input and output means.

If we replaced `"location, mood -> haiku"` with `"a, b -> c"`, the LM would be lost. Let’s try it:

In [15]:
haiku_bot = dspy.Predict("a, b -> c")
result = haiku_bot(a="a quiet library", b="mysterious")
print(result.c)

a quiet, mysterious library


Our model doesn’t know we want a haiku, so it just makes a guess.


Naming is the cheapest optimization in DSPy. A field called `research_request` will produce better completions than one called `request`, with no other changes. Signatures are easy to edit; take advantage.


## Typing your fields yields more reliable programs

We can add more specificity to our task by *typing* our fields using the format `name: type`. For example, the signature `"location, mood, contains_pun: bool -> haiku"` accepts a boolean to indicate whether we want our poem to include a pun:


In [16]:
haiku_bot = dspy.Predict("location, mood, contains_pun: bool -> haiku")
result = haiku_bot(location="a quiet library", mood="mysterious", contains_pun=True)
print(result.haiku)

Silent pages whisper,  
Shadows hide secret tales,  
Books guard mystery.


Inline types instruct DSPy to coerce the LM’s output into the types we ask for, and surface clear warnings when they can’t. This catches a class of silent failures that prompt-only systems hide.

Types also let us communicate structural details that are easier to express in code than in natural language. [Richer types](https://dspy.ai/diving-deeper/signatures-in-depth/) – like Pydantic models, `TypedDicts`, or `dataclasses` – can pack plenty of details that help LMs correctly complete a task.

This is especially helpful when typing output fields. For example, if we wanted to modify our program to generate several haikus we could make our output field name plural and type it as a `list[str]`:


In [17]:
haiku_bot = dspy.Predict("location, mood, n: int -> haikus:list[str]")
result = haiku_bot(
    n=3,
    location="a sunny beach",
    mood="relaxed",
)

print(f"Generated {len(result.haikus)} haikus, here's the first:")
print(result.haikus[0])

Generated 3 haikus, here's the first:
Golden sands whisper,
Waves dance under azure sky,
Relaxation's song.


Once a program accrues several fields or we want to add nuanced instructions, it’s likely time to graduate to a **class-based signature**.